# MNIST Digit Classification — Multi-Scale CNN

A compact convolutional neural network for handwritten digit classification on the [MNIST dataset](http://yann.lecun.com/exdb/mnist/), built from two parallel convolutional branches (2x2 and 4x4 kernels) that are concatenated before pooling and a small fully-connected head.

**Result: 99.18% accuracy on the MNIST test set** (10,000 images), with a model of only ~134K trainable parameters.

This notebook can be used in two ways:
- **Load the pretrained weights** included in this repository (`models/mnist_cnn_model.pth`) and go straight to evaluation — this is the default.
- **Train the model from scratch** by setting `TRAIN_FROM_SCRATCH = True` in the configuration cell below (roughly 10-15 minutes on a GPU).


In [ ]:
import os
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


In [ ]:
# Reproducibility & device configuration
SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Configuration

Set `TRAIN_FROM_SCRATCH` to `True` to retrain the model instead of loading the pretrained checkpoint. If no checkpoint is found at `PRETRAINED_MODEL_PATH`, the notebook automatically falls back to training from scratch.


In [ ]:
# --- Toggle: load pretrained weights vs. train from scratch ---
TRAIN_FROM_SCRATCH = False

PRETRAINED_MODEL_PATH = "models/mnist_cnn_model.pth"

SHOULD_TRAIN = TRAIN_FROM_SCRATCH or not os.path.exists(PRETRAINED_MODEL_PATH)
print(f"Mode: {'training from scratch' if SHOULD_TRAIN else 'loading pretrained weights'}")


## Dataset

MNIST is downloaded automatically via `torchvision.datasets.MNIST` on first run. Training images go through light data augmentation (random rotation, padding + random crop) to improve generalization; test images are only normalized.


In [ ]:
# Dataset
# MNIST returns PIL images by default; transforms are applied below.
full_train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=None)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=None)

# Data augmentation (training) and preprocessing (train/test) pipelines
train_transform_augmented = transforms.Compose([
    transforms.RandomRotation(degrees=15),                # random rotation, -15 to +15 degrees
    transforms.Pad(padding=4),                             # pad by 4 pixels on each side
    transforms.RandomCrop(size=28),                        # randomly crop back to 28x28
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),   # MNIST mean/std
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])


class DatasetWithTransform(torch.utils.data.Dataset):
    """Wraps a base dataset and applies a torchvision transform lazily, on __getitem__."""

    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, i):
        img, label = self.dataset[i]
        if self.transform:
            img = self.transform(img)
        return img, label


In [ ]:
# Display a sample training image
image, label = full_train_dataset[16555]
image = transforms.ToTensor()(image).squeeze(0)

plt.imshow(image, cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()


In [ ]:
# Build the DataLoaders
train_set = DatasetWithTransform(full_train_dataset, transform=train_transform_augmented)
test_set = DatasetWithTransform(test_dataset, transform=val_test_transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

print(f"DataLoaders ready. Training batches: {len(train_loader)}")


## Model architecture — MultiScaleNet

A small multi-branch CNN:

- **Two parallel convolution branches** on the raw 28x28 input: a `2x2` kernel and a `4x4` kernel (with padding=1), each producing 32 channels, followed by BatchNorm and ReLU.
- The two branches are **concatenated** channel-wise (64 channels) and passed through two max-pooling layers (`3x3`/stride 3, then `2x2`/stride 2).
- A **fully-connected head**: `Linear(64*4*4 → 128)` + BatchNorm + ReLU + Dropout(0.2), then `Linear(128 → 10)`.

**Total trainable parameters: 133,578**


In [ ]:
class MultiScaleNet(nn.Module):
    def __init__(self):
        super(MultiScaleNet, self).__init__()
        self.branch2x2 = nn.Conv2d(1, 32, kernel_size=2)
        self.bn1a = nn.BatchNorm2d(32)
        self.branch4x4 = nn.Conv2d(1, 32, kernel_size=4, padding=1)
        self.bn1b = nn.BatchNorm2d(32)

        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=3)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x1 = F.relu(self.bn1a(self.branch2x2(x)))
        x2 = F.relu(self.bn1b(self.branch4x4(x)))

        x = torch.cat((x1, x2), dim=1)
        x = self.pool1(x)
        x = self.pool2(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.bn2(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [ ]:
# Instantiate the model, either loading the pretrained checkpoint or starting from scratch
model = MultiScaleNet().to(device)

if not SHOULD_TRAIN:
    state_dict = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    print(f"Loaded pretrained weights from {PRETRAINED_MODEL_PATH}")
else:
    print("Training a new model from scratch." if TRAIN_FROM_SCRATCH else
          f"No pretrained weights found at {PRETRAINED_MODEL_PATH} — training from scratch.")


In [ ]:
# Model summary
print(f"{'Layer':<20} | {'Parameters':<10}")
print("-" * 35)

total_params = 0
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        params = parameter.numel()
        print(f"{name:<20} | {params:<10,}")
        total_params += params

print("-" * 35)
print(f"{'TOTAL':<20} | {total_params:<10,}")


## Training configuration

- **Loss:** cross-entropy with label smoothing (0.05)
- **Optimizer:** AdamW, lr=1e-3, weight_decay=1e-2
- **Scheduler:** `ReduceLROnPlateau` on validation accuracy (factor 0.5, patience 2)
- **Epochs:** up to 50, with early stopping (patience 10, min_delta 1e-4)
- **Batch size:** 64

> Note: for simplicity, this notebook reuses the test set as the validation set during training (there is no separate held-out validation split for the CNN). Keep that in mind if you plan to reuse this training loop with a different evaluation protocol.


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-2,
)

# Hyperparameters
epochs = 50
patience = 10
min_delta = 0.0001

# Learning-rate scheduler, driven by validation accuracy
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)


In [ ]:
if SHOULD_TRAIN:
    train_losses, val_losses = [], []
    best_val_acc = 0.0
    epochs_no_improve = 0

    print(f"Starting training of MultiScaleNet on {device}...")
    for epoch in range(epochs):
        model.train()
        train_running_loss, train_correct, train_total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss = train_running_loss / len(train_loader)
        train_acc = train_correct / train_total
        train_losses.append(train_loss)

        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_running_loss / len(test_loader)
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"Epoch {epoch+1:02} | Train Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | LR: {current_lr:.6f}")

        scheduler.step(val_acc)

        if val_acc > best_val_acc + min_delta:
            best_val_acc = val_acc
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"Training complete. Best validation accuracy: {best_val_acc*100:.2f}%")
else:
    print("Skipping training — using pretrained weights. Set TRAIN_FROM_SCRATCH = True above to retrain.")


## Evaluation

In [ ]:
# Evaluate on the test set
print("Starting evaluation...")
correct, total = 0, 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the {total} test images: {accuracy:.2f}%")


In [ ]:
# Show a few misclassified examples
incorrect_examples = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        predicted = predicted.cpu()

        incorrect_indices = (predicted != labels).nonzero(as_tuple=True)[0]
        for idx in incorrect_indices:
            img = images[idx].cpu().squeeze(0)
            img = img * 0.3081 + 0.1307  # de-normalize for display
            incorrect_examples.append((img, predicted[idx].item(), labels[idx].item()))
            if len(incorrect_examples) >= 5:
                break
        if len(incorrect_examples) >= 5:
            break

print("Displaying 5 misclassified examples:")
if not incorrect_examples:
    print("No errors found!")
else:
    fig, axes = plt.subplots(1, min(5, len(incorrect_examples)), figsize=(15, 3))
    if len(incorrect_examples) == 1:
        axes = [axes]

    for i, (img, pred, true_label) in enumerate(incorrect_examples):
        if i >= 5:
            break
        ax = axes[i]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"Predicted: {pred}\nTrue: {true_label}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## Results

| Metric | Value |
|---|---|
| Test accuracy | **99.18%** |
| Trainable parameters | 133,578 |
| Training epochs (early stopped) | 16 / 50 |
| Best validation accuracy during training | 99.23% |

These results were obtained with the pretrained checkpoint (`models/mnist_cnn_model.pth`) included in this repository.


In [ ]:
if SHOULD_TRAIN:
    os.makedirs("models", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = f"models/mnist_cnn_model_{timestamp}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")
